# Bone Age Model — Internals, Bias & Training Story

End-to-end documentation of a pediatric bone-age regressor (EfficientNet-B4 + gender fusion, RSNA dataset)
and how it was made **machine/scanner-invariant** while improving accuracy.

This notebook covers:
1. **Model internals** — architecture and parameter layout.
2. **Raw vs bias-normalized images** — what the preprocessing pipeline changes and *why those regions carry machine bias*.
3. **Where machine bias lives** — pseudo-machine clustering of low-level image statistics.
4. **Model attention (Grad-CAM)** — does the model look at bone anatomy or at acquisition artifacts?
5. **Training journey** — the techniques that worked (and the one that backfired).
6. **Bias-aware results** — per-slice error across the whole journey.

### The journey at a glance

| Stage | Overall MAD | Machine bias* | Age bias* | Gender bias* |
|---|---|---|---|---|
| Baseline (raw, single B4) | 8.01 | 3.63 | 14.94 | 0.23 |
| + target z-norm + age-balanced sampler | 7.36 | 1.50 | 11.17 | 0.76 |
| + EMA + gender-embedding (**reverted**) | 7.46 | 3.68 | 15.87 | 0.89 |
| Reverted base + TTA (single model) | 7.24 | 1.51 | 12.39 | 1.56 |
| **4-model ensemble + TTA** | **6.52** | **1.76** | **10.73** | **1.24** |

\* = spread (max-min) of mean *signed* error across slice groups, in months. Lower = less bias.

**Context:** RSNA challenge SOTA is ~3.8-4.5 months MAD (on the official 200-image test set); human inter-radiologist
agreement is ~6-8 months. Our 6.52 (internal validation split) sits at human inter-rater level and, unlike the
accuracy-only SOTA, is explicitly audited for machine/sex invariance.


## 0. Setup

In [ ]:
import os, sys
import numpy as np
import pandas as pd
import torch
import matplotlib.pyplot as plt
from PIL import Image
%matplotlib inline

# Notebook lives in the project root; modules import directly.
sys.path.insert(0, os.getcwd())
import config
from model import BoneAgeModel

device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print('torch', torch.__version__, '| device', device)
print('image size', config.IMG_SIZE, '| pseudo-machine clusters', config.N_MACHINE_CLUSTERS)
print('preprocessing default (PREPROCESS):', config.PREPROCESS)

## 1. Model internals

**Architecture:** an ImageNet-pretrained **EfficientNet-B4** backbone produces a 1792-d feature vector
(global-average-pooled). The patient's **sex** is concatenated as a scalar, and a small head regresses bone age:

```
image -> EfficientNet-B4 -> GAP -> 1792-d
sex (0/1) ----------------------------+
                          concat -> 1793-d -> FC(512) -> BN -> ReLU -> Dropout -> FC(1)
```

The target (bone age in months) is **z-normalized using training-split statistics**; predictions are
un-normalized at inference. Sex is fused late so the same hand can map to different ages for boys vs girls.


In [ ]:
def load_member(name):
    p = config.CHECKPOINT_DIR / name
    if not p.exists():
        print('missing', p, '- using best_model.pth'); p = config.CHECKPOINT_DIR / 'best_model.pth'
    m = BoneAgeModel(pretrained=False).to(device).eval()
    ck = torch.load(p, map_location=device, weights_only=False)
    m.load_state_dict(ck['model_state_dict'])
    return m, ck

model, ck = load_member('best_model_seed43.pth')
target_mean = float(ck.get('target_mean', 0.0)); target_std = float(ck.get('target_std', 1.0))

n_back = sum(p.numel() for p in model.backbone.parameters())
n_head = sum(p.numel() for p in model.head.parameters())
print(f'Backbone params : {n_back/1e6:6.2f} M')
print(f'Head params     : {n_head/1e3:6.1f} K')
print(f'Feature dim     : {model.backbone.num_features}')
print(f'Target z-norm   : mean={target_mean:.1f}  std={target_std:.1f} months')
print(f'Checkpoint      : epoch {ck["epoch"]}  val_mad {ck["val_mad"]:.2f}')
print()
print(model.head)

## 2. Raw vs bias-normalized images

X-rays in this dataset come from many machines/institutions. The **machine signature** lives in:
- **Collimation borders, lead markers, background** — pixels *outside* the hand.
- **Global brightness / contrast / polarity** — detector-specific tone curves.

The preprocessing pipeline (`preprocess.py`) neutralizes these so the model can't key on them:
`polarity correction -> hand crop -> CLAHE -> square pad -> resize`.

Below, for sample hands, we (a) draw the detected hand bounding box and **shade the region that gets discarded**
(the machine-signature pixels), (b) show the final normalized image, and (c) compare intensity histograms before/after
CLAHE (the contrast normalization that erases detector tone differences).

> Note: in the final model we **disabled** this preprocessing — measurements showed machine bias was already small and
> the pipeline was accuracy-neutral. The visualization still shows *what machine signal looks like* and why it was a concern.


In [ ]:
import cv2
from preprocess import _to_gray_array, correct_polarity, apply_clahe, preprocess_xray

def hand_bbox(gray):
    blur = cv2.GaussianBlur(gray, (5, 5), 0)
    _, mask = cv2.threshold(blur, 0, 255, cv2.THRESH_BINARY + cv2.THRESH_OTSU)
    k = cv2.getStructuringElement(cv2.MORPH_ELLIPSE, (15, 15))
    mask = cv2.morphologyEx(mask, cv2.MORPH_CLOSE, k)
    cs, _ = cv2.findContours(mask, cv2.RETR_EXTERNAL, cv2.CHAIN_APPROX_SIMPLE)
    if not cs:
        return None
    return cv2.boundingRect(max(cs, key=cv2.contourArea))

sample_ids = [1377, 1378, 1379]
fig, axes = plt.subplots(len(sample_ids), 4, figsize=(16, 4 * len(sample_ids)))
for r, sid in enumerate(sample_ids):
    path = config.TRAIN_IMG_DIR / f'{sid}.png'
    gray = correct_polarity(_to_gray_array(path))
    bbox = hand_bbox(gray)
    disc = gray.copy()
    overlay = cv2.cvtColor(gray, cv2.COLOR_GRAY2RGB)
    if bbox:
        x, y, w, h = bbox
        shade = overlay.copy(); shade[:] = (255, 0, 0)
        mask = np.ones(gray.shape, bool); mask[y:y+h, x:x+w] = False
        overlay[mask] = (0.6 * overlay[mask] + 0.4 * shade[mask]).astype('uint8')
        cv2.rectangle(overlay, (x, y), (x+w, y+h), (0, 255, 0), 6)
    prep = np.array(preprocess_xray(path, out_size=config.IMG_SIZE).convert('L'))

    axes[r, 0].imshow(gray, cmap='gray'); axes[r, 0].set_title(f'id {sid}: raw (polarity-fixed)')
    axes[r, 1].imshow(overlay); axes[r, 1].set_title('discarded machine-signal region (red)')
    axes[r, 2].imshow(prep, cmap='gray'); axes[r, 2].set_title('bias-normalized (crop+CLAHE+square)')
    axes[r, 3].hist(gray.ravel(), bins=50, alpha=0.5, label='raw')
    axes[r, 3].hist(apply_clahe(gray).ravel(), bins=50, alpha=0.5, label='CLAHE')
    axes[r, 3].set_title('intensity histogram'); axes[r, 3].legend()
    for c in range(3):
        axes[r, c].axis('off')
plt.tight_layout(); plt.show()
print('Red = pixels outside the hand (borders/markers/background) that most strongly fingerprint a machine.')
print('CLAHE flattens the brightness/contrast differences between detectors (histogram becomes more uniform).')

## 3. Where machine bias lives — pseudo-machine clustering

There is no machine label in the dataset, so we derive **pseudo-machine clusters** by KMeans on low-level image
statistics (mean/percentile intensity, dark-fraction, resolution, aspect) of the *raw* images — the acquisition
fingerprint. If clusters differ systematically in these stats, that is exactly the variation a biased model could
exploit. The bias-aware evaluation (`evaluate_bias.py`) then reports error *per cluster* so we can check the model
does **not** depend on them.


In [ ]:
from sklearn.cluster import KMeans
from sklearn.preprocessing import StandardScaler
from dataset import load_dataframes
from evaluate_bias import image_stats

_, val_df = load_dataframes(config.TRAIN_CSV, val_split=config.VAL_SPLIT, seed=config.SEED)
val_df = val_df.reset_index(drop=True)
samp = val_df.sample(min(400, len(val_df)), random_state=0).reset_index(drop=True)

cols = ['mean', 'std', 'p5', 'p50', 'p95', 'frac_dark', 'h', 'w', 'aspect']
F = pd.DataFrame([image_stats(config.TRAIN_IMG_DIR / f'{int(i)}.png') for i in samp['id']], columns=cols)
F['cluster'] = KMeans(n_clusters=config.N_MACHINE_CLUSTERS, random_state=config.SEED,
                      n_init=10).fit_predict(StandardScaler().fit_transform(F.values))

print('Per-cluster acquisition characteristics (means):')
print(F.groupby('cluster')[['mean', 'std', 'p95', 'frac_dark', 'h']].mean().round(1))

fig, ax = plt.subplots(1, 3, figsize=(16, 4))
F.groupby('cluster')['mean'].mean().plot.bar(ax=ax[0], title='mean brightness by cluster')
F.groupby('cluster')['frac_dark'].mean().plot.bar(ax=ax[1], title='dark-background fraction by cluster')
F.groupby('cluster')['h'].mean().plot.bar(ax=ax[2], title='image height (resolution) by cluster')
for a in ax: a.set_xlabel('pseudo-machine cluster')
plt.tight_layout(); plt.show()
print('Clusters separate by brightness/background/resolution => these are scanner/acquisition signatures.')

In [ ]:
# One example image per cluster - visually different acquisition styles
fig, axes = plt.subplots(1, config.N_MACHINE_CLUSTERS, figsize=(3 * config.N_MACHINE_CLUSTERS, 3))
for c in range(config.N_MACHINE_CLUSTERS):
    idx = F.index[F['cluster'] == c]
    ax = axes[c] if config.N_MACHINE_CLUSTERS > 1 else axes
    if len(idx):
        sid = int(samp.loc[idx[0], 'id'])
        ax.imshow(Image.open(config.TRAIN_IMG_DIR / f'{sid}.png').convert('L'), cmap='gray')
    ax.set_title(f'cluster {c}'); ax.axis('off')
plt.tight_layout(); plt.show()

## 4. Model attention (Grad-CAM)

Grad-CAM highlights the image regions that drive the prediction. We want the model to attend to **bones**
(carpals, epiphyses, growth plates) — not borders, markers, or background, which would indicate machine bias.
We overlay attention for the raw image and the bias-normalized image.


In [ ]:
from transforms import get_val_transforms

def gradcam(model, pil_rgb, male):
    tf = get_val_transforms()
    x = tf(pil_rgb).unsqueeze(0).to(device)
    g = torch.tensor([[float(male)]], device=device)
    feat = model.backbone.forward_features(x)   # (1, C, h, w) spatial map
    feat.retain_grad()
    out = model.head(torch.cat([feat.mean(dim=(2, 3)), g], dim=1))  # replicate GAP + head
    model.zero_grad(); out.backward()
    w = feat.grad.mean(dim=(2, 3), keepdim=True)
    cam = (w * feat).sum(1).relu().squeeze().detach().cpu().numpy()
    cam = (cam - cam.min()) / (np.ptp(cam) + 1e-8)
    return cam, float(out.item()) * target_std + target_mean

def overlay_cam(pil_rgb, cam):
    base = np.array(pil_rgb.resize((config.IMG_SIZE, config.IMG_SIZE)).convert('L'))
    heat = cv2.applyColorMap((cv2.resize(cam, (config.IMG_SIZE, config.IMG_SIZE)) * 255).astype('uint8'),
                             cv2.COLORMAP_JET)
    heat = cv2.cvtColor(heat, cv2.COLOR_BGR2RGB)
    return (0.55 * np.stack([base] * 3, -1) + 0.45 * heat).astype('uint8')

sid, male = 1377, 0   # 15-year-old female
raw = Image.open(config.TRAIN_IMG_DIR / f'{sid}.png').convert('RGB')
prep = preprocess_xray(config.TRAIN_IMG_DIR / f'{sid}.png', out_size=config.IMG_SIZE)

fig, axes = plt.subplots(1, 2, figsize=(11, 6))
for ax, (title, img) in zip(axes, [('raw input', raw), ('bias-normalized input', prep)]):
    cam, pred = gradcam(model, img, male)
    ax.imshow(overlay_cam(img, cam)); ax.axis('off')
    ax.set_title(f'{title}\npredicted {pred:.1f} mo (true {int(val_df.loc[val_df.id==sid, "boneage"].iloc[0]) if (val_df.id==sid).any() else "?"})')
plt.tight_layout(); plt.show()
print('Attention concentrating on the hand/wrist bones (not borders) indicates the model uses anatomy, not artifacts.')

## 5. Training journey — what worked and what backfired

Each step was kept or discarded based on the **bias-aware evaluation**, not just overall MAD.

1. **Target z-normalization** — regressing raw months (0-228) made the model hedge toward the mean. Z-scoring the
   target (train-split stats, un-normalized at eval) fixed sluggish convergence.
2. **Age-balanced sampling** — the dataset is heavily imbalanced (16y+ has only ~42 samples). A `WeightedRandomSampler`
   by inverse age-bin frequency directly attacked the regression-to-the-mean at the age extremes. Improved *every*
   age bin with no trade-off (8.01 -> 7.36 MAD).
3. **Bias-normalization preprocessing (CLAHE/crop/polarity)** — built and measured, but it was accuracy-neutral and
   did not reduce machine bias further, so it was **disabled** to keep the pipeline simple.
4. **EMA + learned gender embedding** — *backfired*. With cosine-LR annealed to zero, the raw final weights are
   already the optimum; EMA averaged in worse earlier weights and dragged predictions toward the resampled mean.
   MAD 7.36 -> 7.46 and machine bias 1.50 -> 3.68. **Reverted.**
5. **Test-time augmentation (TTA)** — averaging over flip + small rotations; small, free gain kept at eval time.
6. **4-model seed ensemble** — averaging diverse seeds (42/43/44/45) gave the largest accuracy gain (-> 6.52)
   while keeping bias controlled.

The learning curves below are parsed from the per-member training logs.


In [ ]:
def parse_valmad(path):
    ys = []
    if not os.path.exists(path):
        return ys
    for line in open(path, encoding='utf-8', errors='ignore'):
        if 'Val MAD' in line:
            try:
                ys.append(float(line.split('Val MAD')[1].split(':')[1].split()[0]))
            except Exception:
                pass
    return ys

logs = {'seed42 (base)': 'train_run.log', 'seed43': 'train_seed43.log',
        'seed44': 'train_seed44.log', 'seed45': 'train_seed45.log'}
plt.figure(figsize=(10, 5))
for lbl, f in logs.items():
    ys = parse_valmad(f)
    if ys:
        plt.plot(range(1, len(ys) + 1), ys, marker='.', label=f'{lbl} (best {min(ys):.2f})')
plt.axhline(6.52, ls='--', color='green', label='ensemble 6.52')
plt.xlabel('epoch'); plt.ylabel('Val MAD (months)'); plt.ylim(0, 20)
plt.title('Per-member learning curves (warmup epochs 1-5 = frozen backbone)')
plt.legend(); plt.grid(alpha=0.3); plt.show()

## 6. Bias-aware results across the journey

The key metric for invariance is the **spread of mean signed error across slice groups** (gender / age bin /
pseudo-machine cluster). Lower spread = the model's error does not depend on that attribute. Note how the EMA
experiment spikes both MAD and machine bias (why it was reverted), and how machine + gender bias stay low throughout
while the residual age bias is partly irreducible (few samples + skeletal maturity plateau at 16y+).


In [ ]:
stages  = ['baseline', 'znorm+\nsampler', 'EMA+emb\n(reverted)', 'base+TTA', 'ensemble\n(4x)']
mad     = [8.01, 7.36, 7.46, 7.24, 6.52]
machine = [3.63, 1.50, 3.68, 1.51, 1.76]
age     = [14.94, 11.17, 15.87, 12.39, 10.73]
gender  = [0.23, 0.76, 0.89, 1.56, 1.24]

x = np.arange(len(stages)); w = 0.25
fig, (a1, a2) = plt.subplots(1, 2, figsize=(15, 5))
a1.plot(x, mad, 'o-', color='black')
for xi, yi in zip(x, mad):
    a1.annotate(f'{yi:.2f}', (xi, yi), textcoords='offset points', xytext=(0, 7), ha='center')
a1.set_xticks(x); a1.set_xticklabels(stages); a1.set_ylim(0, 9)
a1.set_title('Overall MAD (months) - lower is better'); a1.grid(alpha=0.3)

a2.bar(x - w, machine, w, label='machine'); a2.bar(x, age, w, label='age'); a2.bar(x + w, gender, w, label='gender')
a2.set_xticks(x); a2.set_xticklabels(stages)
a2.set_title('Signed-error spread (max-min, months) - lower = less bias'); a2.legend(); a2.grid(alpha=0.3)
plt.tight_layout(); plt.show()

## 7. Conclusions

- **Accuracy:** 8.01 -> **6.52 months MAD** (internal validation), via target z-norm + age-balanced sampling + TTA + a
  4-model ensemble — all within a single 12 GB laptop GPU budget.
- **Bias / invariance (the priority):** machine-cluster signed-error spread held at ~1.5-1.8 months and gender at
  ~1.2 — i.e. **predictions do not meaningfully depend on which X-ray machine or the patient's sex.** The cluster
  *MAD* spread is driven by intrinsic difficulty (one harder cluster), not bias (its signed error is ~0).
- **Residual:** age-extreme bias (16y+ under-prediction) is partly irreducible: ~42 samples and a skeletal-maturity
  plateau. An ensemble plus balanced sampling shrank it but cannot remove it.
- **Context:** SOTA on the official RSNA test set is ~3.8-4.5 months but is **accuracy-only** and is known to break
  under brightness/contrast/resolution shifts; human inter-rater agreement is ~6-8 months. Our model sits at
  human inter-rater level *and* is explicitly invariance-audited.

**Next levers** (untried here): RoI hand-detection + higher resolution, larger/multi-architecture ensembles, 5-fold CV,
and replacing the weak seed-42 member (the ensemble of seeds 43/44/45 alone may beat 6.52).
